In [1]:
#%config InlineBackend.figure_formats=['svg']
from src.Utils import plot_traces, save_results, get_means
from src.Dataset import dataset_TES
from src.LoadAutoencoder import load_autoencoder
from src.ExistingAlgorithms import max_value, area, sklearn_available
from src.GaussianMixture import gaussian_mixture

from sklearn.decomposition import PCA, KernelPCA, NMF
from sklearn.manifold import Isomap, TSNE

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Remove outlier cluster
#gm.cluster_means = np.delete(gm.cluster_means, 4,0)
#gm.cluster_covariances = np.delete(gm.cluster_covariances, 4,0)
#gm.cluster_weights = np.delete(gm.cluster_weights, 4,0)

In [ ]:
# Add point 
#mean_ = np.append(get_means(name_method, path = path_results), [get_means(name_method, path = path_results)[10]+0.01], axis=0)
#mean_

In [ ]:
weights = [1.00000000e+00, 8.67567149e-01, 5.11893733e-01, 4.64791825e-01, 
           1.95257413e-01, 3.48166360e-01,1.24623185e-13, 3.89968505e-14, 3.60043421e-01]

#weights = [0.00047741680462955936, 0.0004709397517675953, 0.0010445432139792974, 0.002944736179232991, 0.04200636206501693, 0.0678533708472783, 0.27526434576344766, 0.1658809085942664, 0.5344563925233353, 0.4171218544172766, 1.083876657430484e-16, 0.2504288272925047, 0.9999999999999999]

weights = [0.0003478958721172963, 0.0003456379713743067, 0.0003744276723215052, 2.7560595871623853e-07, 
           5.1048808392493295e-18, 0.28842984940480537, 0.9999999999999999, 0.4483193456686096, 
           0.34462236788026945, 0.42745454124885873, 6.426683431466243e-24, 3.0632312612371824e-29, 0.4770426849308018]

path_save = 'src/Results TES (Uniform)/Low Dimension'
path_results = 'src/Results TES (Uniform)'

data_train, data_test, expected_prob = dataset_TES(weights = weights,
                                                signal_size = 8192,
                                                interval = [0,350],#270],
                                                order_dB = False,
                                                normalize = True,
                                                plot_expected = True,
                                                plot_traces = False,
                                                n_photon_number = 50)

In [ ]:
with plt.style.context('seaborn-v0_8'):
    plt.figure(figsize=(5,3), dpi=200)
    plt.plot(data_train[::10,:200].T, linewidth=0.1, alpha=0.05)
    plt.ylabel("Normalized Voltage (a.u.)")
    plt.xlabel("Time (a.u.)")
    plt.yticks([])
    plt.xticks([])
    plt.show()
    

In [ ]:
plt.savefig('Figures/Trace_NIST.pdf', format="pdf", bbox_inches="tight")

### PIKA

In [ ]:
pk = PIKA(n_iter = 100, n_epoch = 10, N_mean = [2, 3])
pk.run_PIKA(X_test, plot=True)

### Max Value

In [ ]:
X_l_MAX = max_value(data_test, filter=True)

In [ ]:
name_method = 'MAX'
condition = X_l_MAX[:,0] < 0.8
gm = gaussian_mixture(X_l_MAX[condition], 
                    data_test[condition],
                    means_init = get_means(name_method),
                    number_cluster = 14,
                    cluster_iter = 20,
                    info_sweep = 10,
                    plot_sweep = True,
                    dpi = 100)

gm.plot_density(bw_adjust = 0.1)
gm.plot_cluster(plot_kde = False)
gm.plot_traces()
gm.plot_traces_average()

In [ ]:
gm.plot_confidence_1d(axis = 0)
gm.plot_trustworthiness()
#save_results(gm = gm, name_method = name_method, path = path_results)

### Area

In [ ]:
condition = np.max(data_test,axis=1) > 0.08
X_l_AREA  = area(data_test[condition], filter = True, plot_filter = True, threshold = 0.04, cutoff = 0.05)

In [ ]:
name_method = 'AREA'
gm = gaussian_mixture(X_l_AREA,
                      data_test[condition],
                      number_cluster = 29,
                      cluster_iter = 30,
                      means_init = get_means(name_method, path_results),
                      tol = 1e-5,
                      info_sweep = 1,
                      plot_sweep = True,
                      label_shift = 1)

gm.plot_density(bw_adjust = 0.03)
gm.plot_cluster(plot_kde = True, bw_adjust = 1)
gm.plot_traces()
gm.plot_traces_average()

In [ ]:
gm.plot_confidence_1d(expected_prob = expected_prob[1:], axis = 0) 
gm.plot_trustworthiness()
#save_results(gm = gm, name_method = name_method, path = path_results)

### Principal Component Analysis (PCA)

In [ ]:
X_l_PCA = sklearn_available(data_train, data_test, path_save, PCA, n_components=2) 

In [ ]:
name_method = 'PCA'
gm = gaussian_mixture(X_l_PCA,
                      data_test,
                      number_cluster = 26,
                      cluster_iter = 20,
                      means_init = get_means(name_method, path_results),
                      tol = 1e-3,
                      info_sweep = 0,
                      plot_sweep = True)
#gm.cmap  = 'Blues'
gm.plot_density(bw_adjust = 0.05,
                bw_adjust_x = 0.05,
                bw_adjust_y = 0.05)
#gm.plot_cluster(plot_kde = True, bw_adjust = 0.1)
#gm.plot_traces()
#gm.plot_traces_average()

In [ ]:
gm.plot_confidence_1d(expected_prob = expected_prob, axis = 0) 
#gm.plot_confidence_2d(expected_prob = expected_prob)
gm.plot_trustworthiness()
save_results(gm = gm, name_method = name_method, path = path_results)

### Kernel PCA (Polynomial)

In [ ]:
X_l_KPCA_P = sklearn_available(data_train, data_test, path_save, KernelPCA,n_components=2, kernel="poly", gamma=1, degree = 2, coef0 = 0) 

In [ ]:
#name_method = 'KPCA Pol'
#gm = gaussian_mixture(X_l_KPCA_P,
#                      data_test,
#                      number_cluster = 27,
#                      cluster_iter = 20,
#                      means_init = None,#get_means(name_method, path_results),
#                      tol = 1e-3,
#                      info_sweep = 1,
#                      plot_sweep = True)

#gm.plot_density(bw_adjust = 0.1,
#                bw_adjust_x = 0.05,
#                bw_adjust_y = 0.05)
#gm.plot_cluster(plot_kde = True, bw_adjust = 0.1)
#gm.plot_traces()
#gm.plot_traces_average()

In [ ]:
#gm.plot_confidence_1d(axis = 0) 
#gm.plot_confidence_2d()
#gm.plot_trustworthiness()
#save_results(gm = gm, name_method = name_method, path = path_results)

### Kernel PCA (Radial basis function)

In [ ]:
X_l_KPCA_RBF = sklearn_available(data_train, data_test, path_save, KernelPCA, n_components = 2, kernel="rbf", gamma = 0.5) 

In [ ]:
name_method = 'KPCA RBF'
gm = gaussian_mixture(X_l_KPCA_RBF,
                      data_test,
                      number_cluster = 25,
                      cluster_iter = 20,
                      tol = 1e-3,
                      means_init = get_means(name_method, path_results),
                      info_sweep = 1,
                      plot_sweep = True)

gm.plot_density(bw_adjust = 0.1,
                bw_adjust_x = 0.05,
                bw_adjust_y = 0.05)
gm.plot_cluster(plot_kde = True, bw_adjust = 0.1)
#gm.plot_traces()
#gm.plot_traces_average()

In [ ]:
# Remove outlier cluster
#gm.cluster_means = np.delete(gm.cluster_means, 11,0)
#gm.cluster_covariances = np.delete(gm.cluster_covariances, 11,0)
#gm.cluster_weights = np.delete(gm.cluster_weights, 11,0)

In [ ]:
gm.plot_confidence_1d(axis = 0) 
gm.plot_confidence_2d()
gm.plot_trustworthiness()
save_results(gm = gm, name_method = name_method, path = path_results)

### Kernel PCA (Sigmoid)

In [ ]:
X_l_KPCA_S = sklearn_available(data_train, data_test, path_save, KernelPCA,n_components=2,kernel="sigmoid",gamma=0.01, fit_inverse_transform=True) 

In [ ]:
name_method = 'KPCA Sig'
gm = gaussian_mixture(X_l_KPCA_S,
                      data_test,
                      number_cluster = 24,
                      cluster_iter = 20,
                      tol = 1e-5,
                      means_init = get_means(name_method, path_results),
                      info_sweep = 1,
                      plot_sweep = True)

gm.plot_density(bw_adjust = 0.1,
                bw_adjust_x = 0.05,
                bw_adjust_y = 0.05)
gm.plot_cluster(plot_kde = True, bw_adjust = 0.1)
gm.plot_traces()
gm.plot_traces_average()

In [ ]:
# Remove outlier cluster
#gm.cluster_means = np.delete(gm.cluster_means, 5,0)
#gm.cluster_covariances = np.delete(gm.cluster_covariances, 5,0)
#gm.cluster_weights = np.delete(gm.cluster_weights, 5,0)

In [ ]:
gm.plot_confidence_1d(axis = 0) 
gm.plot_confidence_2d()
gm.plot_trustworthiness()
#save_results(gm = gm, name_method = name_method, path = path_results)

### Kernel PCA (Cosine)

In [ ]:
X_l_KPCA_C = sklearn_available(data_train, data_test, path_save, KernelPCA,n_components=2,kernel="cosine",gamma=0.01) 

In [ ]:
name_method = 'KPCA Cos'
gm = gaussian_mixture(X_l_KPCA_C,
                      data_test,
                      number_cluster = 18,
                      cluster_iter = 20,
                      tol = 1e-5,
                      means_init = get_means(name_method, path_results),
                      info_sweep = 1,
                      plot_sweep = True)

gm.plot_density(bw_adjust = 0.1,
                bw_adjust_x = 0.05,
                bw_adjust_y = 0.05)
gm.plot_cluster(plot_kde = True, bw_adjust = 0.1)
gm.plot_traces()
gm.plot_traces_average()

In [ ]:
gm.plot_confidence_1d(axis = 0) 
gm.plot_confidence_2d()
gm.plot_trustworthiness()
save_results(gm = gm, name_method = name_method, path = path_results)

### t-distributed Stochastic Neighbor Embedding (t-SNE)

In [ ]:
n_samples = data_test.shape[0]
perplexity = np.linspace(5, 4*np.sqrt(n_samples), 10)
divergence = []

for i in perplexity:
    model = TSNE(n_components=1, init="pca", perplexity=i)
    reduced = model.fit_transform(data_test)
    divergence.append(model.kl_divergence_)

with plt.style.context("seaborn-v0_8"):
    plt.figure(figsize = (10,4))
    plt.plot(perplexity, divergence)
    plt.ylabel('KL divergence')
    plt.xlabel('Perplexity')
    plt.show()

In [ ]:
X_l_tSNE = sklearn_available(data_train, data_test, path_save, TSNE,n_components=2, perplexity = 450) 

In [ ]:
name_method = 'tSNE'
gm = gaussian_mixture(X_l_tSNE,
                      data_test,
                      number_cluster = 26,
                      cluster_iter = 30,
                      tol = 1e-3,
                      means_init = get_means(name_method, path_results),
                      info_sweep = 0,
                      plot_sweep = True)

gm.plot_density(bw_adjust = 0.22,
                bw_adjust_x = 0.05,
                bw_adjust_y = 0.05)
#gm.plot_cluster(plot_kde = True, bw_adjust = 0.1)
#gm.plot_traces()
#gm.plot_traces_average()

In [ ]:
gm.plot_confidence_1d(axis = 0) 
gm.plot_trustworthiness()
save_results(gm = gm, name_method = name_method, path = path_results)

### Non-Negative Matrix Factorization (NMF)

In [ ]:
X_l_NMF = sklearn_available(data_train, data_test, path_save, NMF,n_components=1, max_iter=400, solver='mu', beta_loss='kullback-leibler', tol=1e-10)

In [ ]:
name_method = 'NMF'
gm = gaussian_mixture(X_low = X_l_NMF,
                      X_high = data_test,
                      number_cluster = 25,
                      cluster_iter = 20,
                      means_init = None,#get_means(name_method, path = path_results),
                      tol = 1e-3,
                      info_sweep = 0,
                      plot_sweep = True)

gm.plot_density(bw_adjust = 0.01,
                bw_adjust_x = 0.05,
                bw_adjust_y = 0.05)
gm.plot_cluster(plot_kde = False, bw_adjust = 0.1)
gm.plot_traces()
gm.plot_traces_average()

In [ ]:
gm.plot_confidence_1d(axis=0)
gm.plot_confidence_2d()
gm.plot_trustworthiness()
save_results(gm = gm, name_method = name_method, path = path_results)

### Isomap

In [ ]:
X_l_ISO = sklearn_available(data_train, data_test, path_save, Isomap, n_neighbors=500, n_components=1, eigen_solver="arpack")

In [ ]:
name_method = 'ISO'
gm = gaussian_mixture(X_low = X_l_ISO,
                      X_high = data_test,
                      number_cluster = 27,
                      cluster_iter = 20,
                      means_init = None,#get_means(name_method, path = path_results),
                      tol = 1e-5,
                      info_sweep = 0,
                      plot_sweep = True)

gm.plot_density(bw_adjust = 0.05,
                bw_adjust_x = 0.05,
                bw_adjust_y = 0.05)
gm.plot_cluster(plot_kde = False, bw_adjust = 0.1)
gm.plot_traces()
#gm.plot_traces_average()
#kd.plot_traces_average(data_test)

In [ ]:
gm.plot_confidence_1d(axis=0)
gm.plot_trustworthiness()
save_results(gm = gm, name_method = name_method, path = path_results)

### Autoencoder



In [ ]:
path_model = '/home/nicolasdc/files/Photon-Number-Classification/src/AutoencoderLog/grid-search-TES 2D CNN 8/run 18/fold 1'#grid-search-TES 2D CNN 4/run 2/fold 0'
X_i_AUTO, X_r_AUTO, X_l_AUTO = load_autoencoder(data_test, f'{path_model}', filter=False, flip=True) #17, 18, 19, 34-2
X_l_AUTO.shape

In [ ]:
plot_traces(X_r_AUTO)

In [ ]:
name_method = 'Autoencoder'
gm = gaussian_mixture(X_l_AUTO,
                      data_test,
                      number_cluster = 27,
                      cluster_iter = 1,
                      #means_init = get_means(name_method),
                      tol = 1e-4,
                      info_sweep = 0,
                      plot_sweep = True)

gm.plot_density(bw_adjust = 0.1,
                bw_adjust_x = 0.05,
                bw_adjust_y = 0.05)
gm.plot_cluster(plot_kde = True, bw_adjust = 0.1)
#gm.plot_traces()
#gm.plot_traces_average()

In [ ]:
gm.plot_confidence_1d(expected_prob = expected_prob, axis = 0) 
gm.plot_confidence_2d(expected_prob = expected_prob)
gm.plot_trustworthiness()
#save_results(gm = gm, name_method = name_method)

In [ ]:
X_i_AUTOF, X_r_AUTOF, X_l_AUTOF = loadAutoencoder(np.copy(X_test), f'AutoencoderLog/{path_model}', 
                                                  filter=True, 
                                                  threshold=0.003)

In [ ]:
import matplotlib.pyplot as plt


plt.figure()
plt.scatter(X_l_AUTOF[:,0],X_l_AUTOF[:,1], s=1, alpha=0.01)#, c=X_l_AREA)
plt.show()

In [ ]:
kd = gaussian_mixture(X_l_AUTO, 
                    number_cluster=18,
                    flip = True)

kd.plot_cluster()
kd.plot_psn()
kd.plot_pns()
kd.plot_confidence()

In [ ]:
plt.plot(kd.cluster_means[:14])
plt.show()

### Parametric t-SNE

In [ ]:
import os
import torch

name = 'model_TES' 

# Weights of a torch model
model_weights = os.path.join(f'runs/run 17/models/2 0.003 500.pt')#ParametricTSNE/', name) + ".pt"
loaded_model = torch.load(model_weights).to(device='cpu')

#data_train_ = torch.from_numpy(data_train).float().view(-1, 350).to(device='cuda') 
data_test_ = torch.from_numpy(data_test).float().view(-1, 350).to(device='cpu')

#data_train_ = (data_train_ - data_train_.min()) / (data_train_.max() - data_train_.min())
data_test_ = (data_test_ - data_test_.min()) / (data_test_.max() - data_test_.min())

loaded_model.eval()
with torch.no_grad():
    X_l_PTSNE, X_r_PTSNE = loaded_model(data_test_, both = True)
    X_l_PTSNE, X_r_PTSNE = X_l_PTSNE.cpu().numpy(), X_r_PTSNE.cpu().numpy()

In [ ]:
name_method = 'PTSNE'
gm = gaussian_mixture(X_low = X_l_PTSNE,
                      X_high = data_test,
                      number_cluster = 25, 
                      cluster_iter = 1,
                      means_init = None,#get_means(name_method, path = path_results),
                      tol = 1e-5,
                      info_sweep = 0,
                      plot_sweep = True )

gm.plot_density(bw_adjust = 0.01,#22,
                bw_adjust_x = 0.05,
                bw_adjust_y = 0.05)
gm.plot_cluster(plot_kde = False, bw_adjust = 0.1)
gm.plot_traces()
gm.plot_traces_average()

In [ ]:
gm.plot_confidence_2d()
gm.plot_trustworthiness()
save_results(gm = gm, name_method = name_method, path = path_results)